In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hongweicao/catanddogsmall")

print("Path to dataset files:", path)

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\hongweicao\catanddogsmall\versions\1


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16,ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense,Flatten,Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib as plt


In [7]:
train_path=path+"/dogvscat_small/train"
testing_path=path+"/dogvscat_small/test"
validation_path=path+"/dogvscat_small/validation"

In [8]:
train_data_generator=ImageDataGenerator(
    rescale=1/255,
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

validation_data_generator=ImageDataGenerator(rescale=255)

In [10]:
train_generator=train_data_generator.flow_from_directory(train_path,target_size=(224,224),batch_size=32,class_mode="binary")
validation_generator=validation_data_generator.flow_from_directory(validation_path,target_size=(224,224),batch_size=32,class_mode="binary")
test_generator=validation_data_generator.flow_from_directory(testing_path,target_size=(224,224),batch_size=32,class_mode="binary")

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [11]:
base_model=VGG16(weights="imagenet",include_top=False,input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable=False
x=base_model.output
x=Flatten()(x)
x=Dense(128,activation="relu")(x)
x=Dropout(0.5)(x)
predictions=Dense(1,activation="sigmoid")(x)


In [12]:
model=Model(inputs=base_model.input,outputs=predictions)
model.compile(optimizer="Adam",loss="binary_crossentropy",metrics=["Accuracy"])
fitted_model=model.fit(train_generator,epochs=5,validation_data=validation_generator)
loss,accuracy=model.evaluate(test_generator)
print(loss)
print(accuracy)

Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 193s 3s/step - Accuracy: 0.7035 - loss: 0.9024 - val_Accuracy: 0.8400 - val_loss: 1754.3461
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 194s 3s/step - Accuracy: 0.8270 - loss: 0.3843 - val_Accuracy: 0.8660 - val_loss: 2159.6812
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 190s 3s/step - Accuracy: 0.8440 - loss: 0.3616 - val_Accuracy: 0.8780 - val_loss: 2394.8188
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 187s 3s/step - Accuracy: 0.8620 - loss: 0.3185 - val_Accuracy: 0.8890 - val_loss: 2590.0139
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 185s 3s/step - Accuracy: 0.8540 - loss: 0.3294 - val_Accuracy: 0.8890 - val_loss: 2467.5383
32/32 ━━━━━━━━━━━━━━━━━━━━ 60s 2s/step - Accuracy: 0.8740 - loss: 2624.4868
2624.48681640625
0.8740000128746033


In [13]:
image_path=validation_path+"\\dogs\\1009.jpg"
img=image.load_img(image_path,target_size=(224,224))
image_array=image.img_to_array(img)
image_array=np.expand_dims(image_array,axis=0)
image_array=image_array/255
prediction=model.predict(image_array)
print(prediction)
prediction= "cat" if prediction[0]<0.5 else "dog"
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step
[[0.9900822]]
dog


In [ ]:
images,labels=next(validation_generator)
final_prediction=model.predict(images)
plt.figure(figsize=(12,12))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    actual_label= "cat" if labels[i]<0.5 else "dog"
    predicted_label= "cat" if final_prediction[i]<0.5 else "dog"
    plt.title(f"Actual: {actual_label}, Predicted:{predicted_label}")
    plt.axis("off")
plt.show()



1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


TypeError: 'module' object is not callable